# Evaluation of extremes of models on 60km -> 2.2km-4x over Birmingham

In [ ]:
%reload_ext autoreload

%autoreload 2

%reload_ext dotenv
%dotenv

In [ ]:
from mlde_analysis.default_params import *

In [ ]:
import functools
import math
import string

import cf_xarray
import IPython
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from mlde_analysis import plot_map
from mlde_analysis.display import pretty_table, VAR_RANGES
from mlde_analysis.distribution import plot_freq_density_figure, compute_metrics, DIST_THRESHOLDS
from mlde_utils import cp_model_rotated_pole, platecarree
from mlde_analysis import qq_plot, reasonable_quantiles

In [ ]:
matplotlib.rcParams['figure.dpi'] = 300

In [ ]:
IPython.display.Markdown(desc)

In [ ]:
%reload_ext mlde_analysis.magics 
EVAL_DS, MODELS, CPM_DAS, PRED_DAS, VAR_DAS, MODELLABEL2SPEC = %load_eval_data
EVAL_DS

In [ ]:
box_queries = {
    "London": cp_model_rotated_pole.transform_point(-0.118092, 51.509865, src_crs=platecarree) + np.array([360, 0]),  # for rotated pole, longitude runs over 360
    "Birmingham": cp_model_rotated_pole.transform_point(-1.898575, 52.489471, src_crs=platecarree) + np.array([360, 0]),  # for rotated pole, longitude runs over 360
}


fig = plt.figure(layout="constrained", figsize=(1.5, 1.5))
ax = fig.subplots(subplot_kw={"projection": cp_model_rotated_pole})
ax.coastlines(**{"resolution": "10m", "linewidth": 0.3})
da = CPM_DAS[eval_vars[0]]
ax.set_extent((
    da.cf["X"].min(),
    da.cf["X"].max(),
    da.cf["Y"].min(),
    da.cf["Y"].max(),
))

for label, q in box_queries.items():
    single_box_da = da.cf.sel(X=q[0], Y=q[1], method="nearest")
    ax.plot(single_box_da.cf["X"].values, single_box_da.cf["Y"].values, color='blue', markersize=0.5, marker='o', transform=cp_model_rotated_pole)
    ax.annotate(
        xy=(single_box_da.cf["X"].item(), single_box_da.cf["Y"].item()), xycoords="data",
        text=label, xytext=(0, -15), textcoords="offset pixels",
        ha='center',va="center", transform=cp_model_rotated_pole, fontsize="xx-small")
    
plt.show()

## Figure: single box distribution

* Frequency Density Histogram of rainfall intensities

Table of:

* RMS biases
* J-S Distances
* proportion of density over thresholds

In [ ]:
def _metrics(label, q, var_ds, thresholds):
    ds = var_ds.cf.sel(X=q[0], Y=q[1], method="nearest")
    cpm_da = ds[f"target_{var}"]
    pred_da = ds[f"pred_{var}"]
    
    return compute_metrics(pred_da, cpm_da, thresholds=thresholds).expand_dims({"location": [label]})
    
for var in eval_vars:
    IPython.display.display_markdown(f"### {var}", raw=True)
        
    metrics_ds = xr.concat(
        [ _metrics(label, q, VAR_DAS[var], DIST_THRESHOLDS[var]) for label, q in box_queries.items() ], 
        dim="location"
    )

    pretty_table(metrics_ds, round=4)
    
    for label, q in box_queries.items():
        ds = VAR_DAS[var].cf.sel(X=q[0], Y=q[1], method="nearest")
        pred_da = ds[f"pred_{var}"]
        cpm_da = ds[f"target_{var}"]
        
        fig = plt.figure(layout="constrained", figsize=(3.5, 2.5))
        
        ax = plot_freq_density_figure(pred_da, cpm_da, MODELLABEL2SPEC, fig)
        
        ax.axvline(x=cpm_da.max(), color='k', linestyle='--', linewidth=1)
        ax.set_title(label)
        
        plt.show()

## Figure: per time period distribution

* Frequency Density Histogram of rainfall intensities

Table of:

* RMS biases
* J-S Distances
* proportion of density over thresholds

In [ ]:
for var in eval_vars:
    IPython.display.display_markdown(f"### {var}", raw=True)
    
    metrics_ds = xr.concat([
            xr.concat([ _metrics(label, q, tp_ds, DIST_THRESHOLDS[var]) for label, q in box_queries.items() ], dim="location").expand_dims({"time_period": [tp]})
         for tp, tp_ds in VAR_DAS[var].groupby("time_period") ], 
        dim="time_period"
    )

    pretty_table(metrics_ds, round=4)
    
    for label, q in box_queries.items():
        ds = VAR_DAS[var].cf.sel(X=q[0], Y=q[1], method="nearest")
        for tp, tp_ds in ds.groupby("time_period"):
            pred_da = tp_ds[f"pred_{var}"]
            cpm_da = tp_ds[f"target_{var}"]
    
            fig = plt.figure(layout="constrained", figsize=(3.5, 2.5))
            ax = plot_freq_density_figure(pred_da, cpm_da, MODELLABEL2SPEC, fig)
            ax.axvline(x=cpm_da.max(), color='k', linestyle='--', linewidth=1)
            ax.set_title(f"{label} {tp}")
            
            plt.show()

## QQ plots

In [ ]:
quantile_dims=["ensemble_member", "time"]

for var in eval_vars:
    IPython.display.display_markdown(f"### {var}", raw=True)
    for label, q in box_queries.items():
        ds = VAR_DAS[var].cf.sel(X=q[0], Y=q[1], method="nearest")
        pred_da = ds[f"pred_{var}"]
        cpm_da = ds[f"target_{var}"]
        
        quantiles = reasonable_quantiles(cpm_da)
        cpm_quantiles = cpm_da.quantile(quantiles, dim=quantile_dims).rename("target_q")
    
        pred_quantiles = pred_da.quantile(quantiles, dim=quantile_dims).rename("pred_q")

        layout="constrained"

        fig, ax = plt.subplots(figsize=(5.5, 5.5), layout="constrained")

        xlabel = f"CPM \n{xr.plot.utils.label_from_attrs(da=cpm_da)}"
        ylabel = f"Predicted \n{xr.plot.utils.label_from_attrs(da=pred_da)}"

        qq_plot(ax, cpm_quantiles, pred_quantiles, title=f"Predicted quantiles vs CPM quantiles", xlabel=xlabel, ylabel=ylabel)

        plt.show()